## I maxed out my free colab computing units from last notebook so I had to copy and paste and adjust some code to pick off where I left off (using a separate email).


In [2]:
import zipfile
import io
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
zip_path = '/content/drive/My Drive/evaluated_positions.zip'
extract_path = '/content/extracted_positions'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
!pip install chess

In [6]:
"""
Convert FEN strings to tensor representations for neural network input.
"""

import chess
import numpy as np
import torch

import json

class ChessDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_path, use_attack_map=False):
        self.use_attack_map = use_attack_map
        self.data_entries = []

        with open(jsonl_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.data_entries.append(json.loads(line))

    def __len__(self):
        return len(self.data_entries)

    def __getitem__(self, idx):
        entry = self.data_entries[idx]

        # Generate the features
        X_tensor = fen_to_tensor(entry['fen'], self.use_attack_map)

        # Extract the target label as a float32 tensor
        y_target = torch.tensor(entry['target'], dtype=torch.float32)

        return X_tensor, y_target

def fen_to_tensor(fen, use_attack_map=False):
    board = chess.Board(fen)
    channels = 22 if use_attack_map else 20
    tensor = np.zeros((channels, 8, 8), dtype=np.float32)
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is not None:
            color_idx = 0 if piece.color == chess.WHITE else 1
            channel = (piece.piece_type - 1) + (color_idx * 6)
            rank, col = divmod(square, 8)
            row = 7 - rank
            tensor[channel, row, col] = 1.0

    tensor[12, :, :] = int(board.turn)  # Turn channel
    tensor[13, :, :] = 1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0  # White kingside castling
    tensor[14, :, :] = 1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0  # White queenside castling
    tensor[15, :, :] = 1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0  # Black kingside castling
    tensor[16, :, :] = 1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0  # Black queenside castling

    if board.ep_square is not None:
        ep_row, ep_col = divmod(board.ep_square, 8)
        tensor[17, ep_row, ep_col] = 1.0  # En passant square
    tensor[18, :, :] = board.halfmove_clock / 100.0  # Halfmove clock
    tensor[19, :, :] = min(board.fullmove_number, 200) / 200.0  # Fullmove number

    if use_attack_map:
        attack_map = np.zeros((2, 8, 8), dtype=np.float32)
        for square in chess.SQUARES:
            rank, col = divmod(square, 8)
            row = 7 - rank

            attack_map[0, row, col] = 1.0 if board.attackers(chess.WHITE, square) else 0.0
            attack_map[1, row, col] = 1.0 if board.attackers(chess.BLACK, square) else 0.0
        tensor[20:22, :, :] = attack_map

    return torch.tensor(tensor)



In [7]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlock, self).__init__()
        # not for this resnet we are not downsampling, so stride is always 1

        # conv layer 1
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # conv layer 2
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()


    def forward(self, x):
        residual = x
        # pass through layer 1
        out = self.relu(self.bn1(self.conv1(x)))
        # pass through layer 2
        out = self.bn2(self.conv2(out))
        # residual connection
        out += self.shortcut(residual)
        # apply ReLU activation
        out = self.relu(out)

        return out


class ChessResNet(nn.Module):
    def __init__(self, input_channels=20, num_blocks=5, num_classes=1):
        super().__init__()
        self.input_channels = input_channels
        self.num_blocks = num_blocks
        self.num_classes = num_classes

        # Initial convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        # Residual blocks
        self.res_blocks = self._make_layer(64, 64, num_blocks)

        # Compressing values with 1x1 convolution to reduce the number of channels before flattening
        self.value_conv = nn.Conv2d(64, 2, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(2)
        self.relu_value = nn.ReLU(inplace=True)

        # Fully connected layer for output
        self.fc = nn.Sequential(
            nn.Linear(2 * 8 * 8, 64),  # Flattened size after conv layers
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
            nn.Sigmoid()  # Use Sigmoid for binary output (0 to 1)
        )

    def _make_layer(self, in_channels, out_channels, num_blocks):
        layers = [ResBlock(in_channels, out_channels)]
        for _ in range(1, num_blocks):
            layers.append(ResBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        # Initial convolutional layer
        out = self.relu(self.bn1(self.conv1(x)))

        # Pass through residual blocks
        out = self.res_blocks(out)

        # Compressing values with 1x1 convolution
        out = self.relu_value(self.value_bn(self.value_conv(out)))

        # Flatten the output for the fully connected layer
        out = out.view(out.size(0), -1)

        # Fully connected layers
        out = self.fc(out)

        return out

In [8]:
from torch.utils.data import random_split, DataLoader

In [8]:
import torch
from torch.utils.data import DataLoader, random_split

# 1. Force absolute reproducibility across the entire script
SEED = 42
torch.manual_seed(SEED)
generator1 = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessResNet(input_channels=20).to(device)
dataset = ChessDataset("/content/extracted_positions/evaluated_positions.jsonl", use_attack_map=False)

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator1
)

batch_size = 4096
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)

In [ ]:
import torch
import os
from torch.utils.data import DataLoader, random_split

# 1. Establish identical reproducibility seeds
SEED = 42
torch.manual_seed(SEED)
generator1 = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Re-instantiate the Memory-Mapped Dataset and Splits
dataset = ChessMmapDataset(num_positions=10000000)

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=generator1
)

# 3. Establish your exact active loaders
batch_size = 2048
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)

# 4. Initialize Network Architecture
model = ChessResNet(input_channels=20).to(device)

# 5. LOAD CACHED WEIGHTS FROM EPOCH 8
# (Make sure your shared shortcut directory path matches below)
checkpoint_path = "/content/drive/MyDrive/ChessAI/chess_resnet_epoch_8.pt"

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("💪 Successfully loaded Epoch 8 weights from Google Drive!")
else:
    raise FileNotFoundError(f"Could not find your weights file at {checkpoint_path}. Check your Drive mount path!")

# 6. Setup Hyperparameters and Optimization Curve
epochs = 20
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.015,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    anneal_strategy='cos'
)

# 7. THE CRITICAL FAST-FORWARD STEP
# We skip exactly 8 full epochs worth of optimization steps to align the cosine curve
completed_epochs = 9
total_steps_to_skip = completed_epochs * len(train_loader)

print(f"⏩ Fast-forwarding learning rate scheduler past {total_steps_to_skip:,} steps...")
for _ in range(total_steps_to_skip):
    scheduler.step()

# --- RESUME RESNET TRAINING LOOP ---
print(f"🚀 Resuming training pipeline starting at Epoch {completed_epochs + 1}...")

for k in range(completed_epochs, epochs):
    running_loss = 0.0
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1), targets)
        loss.backward()

        optimizer.step()
        scheduler.step()  # Continues along the perfect middle slope of the cosine curve

        running_loss += loss.item()
        if i % 250 == 249:
            print(f"[{k + 1}, {i + 1}] loss: {running_loss / 250:.6f}")
            running_loss = 0.0

    print(f"Finished epoch {k + 1}")

    # Save the new checkpoints back to Drive
    torch.save(model.state_dict(), f"/content/drive/MyDrive/ChessAI/chess_resnet_epoch_{k + 1}.pt")
    print(f"Epoch {k + 1} complete. Weights cached to Drive.")

print("🎉 Finished Training Remaining Epochs!")

In [ ]:
print(len(train_loader), len(val_loader))

1954 245


In [ ]:
# hyper params
epochs = 20
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.02,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    anneal_strategy='cos'
)

In [ ]:
# Training loop
for k in range(epochs):
    running_loss = 0.0
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        # zero the gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(inputs)

        # compute loss
        loss = criterion(outputs.view(-1), targets)

        #backpropagation
        loss.backward()

        # update weights
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        if i % 500 == 499:  # print every 1000 mini-batches
            print(f"[{k + 1}, {i + 1}] loss: {running_loss / 500:.6f}")
            running_loss = 0.0
    print(f"Finished epoch {k + 1}")\
    # save weights every epoch
    torch.save(model.state_dict(), f"/content/drive/MyDrive/ChessAI/chess_resnet_epoch_{k+1}.pt")
    print(f"Epoch {k+1} complete. Weights cached to Drive.")
print("Finished Training")

[1, 500] loss: 0.007370
[1, 1000] loss: 0.005635
[1, 1500] loss: 0.005235
Finished epoch 1
Epoch 1 complete. Weights cached to Drive.
[2, 500] loss: 0.004890
[2, 1000] loss: 0.004731
[2, 1500] loss: 0.004672
Finished epoch 2
Epoch 2 complete. Weights cached to Drive.
[3, 500] loss: 0.004493
[3, 1000] loss: 0.004427
[3, 1500] loss: 0.004391
Finished epoch 3
Epoch 3 complete. Weights cached to Drive.
[4, 500] loss: 0.004261
[4, 1000] loss: 0.004189
[4, 1500] loss: 0.004213
Finished epoch 4
Epoch 4 complete. Weights cached to Drive.
[5, 500] loss: 0.004110
[5, 1000] loss: 0.004091
[5, 1500] loss: 0.004040
Finished epoch 5
Epoch 5 complete. Weights cached to Drive.
[6, 500] loss: 0.003943
[6, 1000] loss: 0.003952
[6, 1500] loss: 0.003881
Finished epoch 6
Epoch 6 complete. Weights cached to Drive.
[7, 500] loss: 0.003754
[7, 1000] loss: 0.003753
[7, 1500] loss: 0.003696
Finished epoch 7
Epoch 7 complete. Weights cached to Drive.
[8, 500] loss: 0.003633
[8, 1000] loss: 0.003592
[8, 1500] los

In [11]:
model.eval()
total_val_loss = 0.0
num_batches = 0
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs.view(-1), targets)
        total_val_loss += loss.item()
        num_batches += 1


print(f"Total validation loss: {total_val_loss / num_batches:.6f}")

Total validation loss: 0.002607


In [ ]:
# loss with use_attack_map= False: 0.00564, train: 0.0049
# loss with use_attack_map= True:0.00562, train:0.0040

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

390727


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

# 1. Enforce identical random seeds to ensure data splits match perfectly
SEED = 42
torch.manual_seed(SEED)
generator1 = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Re-instantiate the model structure (20 input channels)
model = ChessResNet(input_channels=20).to(device)

# 3. Load the completely finalized weights file from your Drive shortcut
# Since the run cut out during Epoch 10, Epoch 9 was fully finished and cached.
checkpoint_path = "/content/drive/MyDrive/ChessAI/chess_resnet_epoch_9.pt"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("💪 Successfully loaded Epoch 9 weights from Google Drive!")

# 4. Re-instantiate your original dataset parsing layout
dataset = ChessDataset("/content/extracted_positions/evaluated_positions.jsonl", use_attack_map=False)

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator1 # Recovers the exact training/validation allocations
)

# 5. Lock in the batch size that successfully cleared memory limits
batch_size = 4096
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, shuffle=False, pin_memory=True)

# 6. Rebuild hyperparameter schedules
epochs = 20
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.02,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    anneal_strategy='cos'
)

# 7. THE FAST-FORWARD MECHANISM
# Skip exactly 9 full epochs of optimization steps to catch the scheduler up to Epoch 10
completed_epochs = 9
total_steps_to_skip = completed_epochs * len(train_loader)

print(f"⏩ Fast-forwarding OneCycleLR scheduler past {total_steps_to_skip:,} steps...")
for _ in range(total_steps_to_skip):
    scheduler.step()

💪 Successfully loaded Epoch 9 weights from Google Drive!
⏩ Fast-forwarding OneCycleLR scheduler past 17,586 steps...


/tmp/ipykernel_3923/883973396.py:60: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


In [10]:
# --- RESUME RESNET TRAINING LOOP ---
print(f"🚀 Resuming production training starting at Epoch {completed_epochs + 1}...")

for k in range(completed_epochs, epochs):
    running_loss = 0.0
    model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1), targets)
        loss.backward()

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        if i % 500 == 499:
            print(f"[{k + 1}, {i + 1}] loss: {running_loss / 500:.6f}")
            running_loss = 0.0

    print(f"Finished epoch {k + 1}")

    # Save checkpoints safely to Google Drive
    torch.save(model.state_dict(), f"/content/drive/MyDrive/ChessAI/chess_resnet_epoch_{k + 1}.pt")
    print(f"Epoch {k + 1} complete. Weights cached to Drive.")

print("🎉 Finished Training Remaining Epochs!")

🚀 Resuming production training starting at Epoch 10...
[10, 500] loss: 0.003389
[10, 1000] loss: 0.003366
[10, 1500] loss: 0.003344
Finished epoch 10
Epoch 10 complete. Weights cached to Drive.
[11, 500] loss: 0.003269
[11, 1000] loss: 0.003257
[11, 1500] loss: 0.003245
Finished epoch 11
Epoch 11 complete. Weights cached to Drive.
[12, 500] loss: 0.003140
[12, 1000] loss: 0.003151
[12, 1500] loss: 0.003128
Finished epoch 12
Epoch 12 complete. Weights cached to Drive.
[13, 500] loss: 0.003016
[13, 1000] loss: 0.002993
[13, 1500] loss: 0.003021
Finished epoch 13
Epoch 13 complete. Weights cached to Drive.
[14, 500] loss: 0.002906
[14, 1000] loss: 0.002911
[14, 1500] loss: 0.002867
Finished epoch 14
Epoch 14 complete. Weights cached to Drive.
[15, 500] loss: 0.002770
[15, 1000] loss: 0.002773
[15, 1500] loss: 0.002735
Finished epoch 15
Epoch 15 complete. Weights cached to Drive.
[16, 500] loss: 0.002618
[16, 1000] loss: 0.002618
[16, 1500] loss: 0.002589
Finished epoch 16
Epoch 16 complet

In [12]:
# train loss: 0.002129
# val loss: 0.002607